# Do director, writer, and cast credits improve recommendations?

This notebook adds people-based content signals to item-based collaborative filtering (CF)
and the CF + plot-text hybrid. It measures whether credits retrieve more of each user's later likes.

- **Prerequisites:** run [01](01_data_exploration.ipynb) for the canonical split and
  [04](04_plot_text_preparation.ipynb) for the plot catalogs. Use the project environment.
- **Data:** official [IMDb non-commercial datasets](https://data.imdb.com/non-commercial-datasets/),
  joined through MovieLens `links.csv`. The preparation cell streams about 1.2 GB on first use
  and saves only matching subsets; reruns verify and reuse them. Internet is needed only for a refresh.
  Observe IMDb's linked usage terms. No API key is needed.
- **Protocol:** same per-user chronological 60/20/20 split, rating >= 4 as a like,
  training-observed candidate movies, and exclusion of **all** movies rated in training.
  Every validation positive remains in Recall@10's denominator, even if unavailable as a candidate.
- **Scope:** validation exploration only. This notebook never reads test feedback, refits on
  validation, or changes notebook 07's frozen selection. Its winner is provisional.
- **Outputs:** coverage, complete tuning grid, per-user metrics, recommendations, and provenance
  under the split-specific `notebook09_credits/` directory.

Credits are movie metadata, not new rating labels. A current IMDb snapshot and the CMU snapshot
do not establish historical availability at every rating timestamp. This remains a retrospective benchmark.

In [1]:
from pathlib import Path
from time import perf_counter
import hashlib
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy.linalg import cho_factor, cho_solve
from scipy.special import expit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from threadpoolctl import threadpool_limits

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10
SEED = 42

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()), None
)
if project_root is None:
    raise FileNotFoundError("Run from inside the MovieLens project.")
raw_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
manifest_path = processed_dir / "split_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 01 first to create the canonical split.")
split_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert split_manifest["split_id"] == SPLIT_ID
assert split_manifest["positive_threshold"] == POSITIVE_THRESHOLD
assert split_manifest["top_k"] == TOP_K

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in split_manifest["raw_sha256"].items():
    if file_sha256(raw_dir / filename) != expected_hash:
        raise ValueError(f"{filename} changed; regenerate the split in notebook 01.")

movies = pd.read_csv(raw_dir / "movies.csv")
loaded_splits = {}
for name in ("train", "validation"):
    path = processed_dir / f"{name}_ratings.csv"
    expected = split_manifest["files"][path.name]
    if file_sha256(path) != expected["sha256"]:
        raise ValueError(f"{path.name} differs from the canonical split.")
    loaded_splits[name] = pd.read_csv(path, parse_dates=["rated_at"])
    assert len(loaded_splits[name]) == expected["rows"]
train_ratings = loaded_splits["train"]
validation_ratings = loaded_splits["validation"]
assert not train_ratings.duplicated(["userId", "movieId"]).any()
assert not validation_ratings.duplicated(["userId", "movieId"]).any()
assert set(map(tuple, train_ratings[["userId", "movieId"]].to_numpy())).isdisjoint(
    map(tuple, validation_ratings[["userId", "movieId"]].to_numpy())
)
assert (train_ratings.groupby("userId")["timestamp"].max()
        <= validation_ratings.groupby("userId")["timestamp"].min()).all()

output_dir = processed_dir / "notebook09_credits"
output_dir.mkdir(parents=True, exist_ok=True)

def with_provenance(frame):
    saved = frame.copy()
    for column, value in {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }.items():
        saved[column] = value
    return saved

def check_provenance(frame, label):
    for column, value in {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }.items():
        assert column in frame and frame[column].eq(value).all(), (
            f"{label}: incompatible {column}; rerun its source notebook."
        )

print("Python:", platform.python_version())
print("Split:", SPLIT_ID)
print("Training / validation ratings:", len(train_ratings), "/", len(validation_ratings))
print("Outputs:", output_dir)

Python: 3.14.10
Split: per_user_chronological_60_20_20_v1
Training / validation ratings: 60255 / 20164
Outputs: C:\Users\clark\OneDrive\Documents\Projects\movie-recommendation-system\data\processed\per_user_chronological_60_20_20_v1\notebook09_credits


## 1. Prepare exact-ID credits and inspect coverage

The helper preserves MovieLens rows with missing credits and checks source/subset hashes.
Directors and writers come from IMDb's title-level crew lists. Writer roles (screenplay, story,
adaptation) are not distinguished in that source. Cast uses the first five **distinct actor/actress
principal credits ordered by IMDb's `ordering` field**. This is a reproducible proxy for prominent cast,
not a verified lead-role or screen-time annotation. Principal-credit coverage can omit performers.

People are represented by IDs, with roles kept separate. Names are used only for readable examples.
The downloaded catalog can include held-out movies; feature fitting and scoring below use only
training candidates. If Windows HTTPS certificate verification fails, install `truststore` in the
environment; the helper uses it when available and never disables verification.

In [2]:
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from scripts.prepare_imdb_credits import prepare_credits

credits_path = prepare_credits(project_root)
credits_source_manifest = json.loads(
    (credits_path.parent / "credits_manifest.json").read_text(encoding="utf-8")
)
assert credits_source_manifest["links_sha256"] == file_sha256(raw_dir / "links.csv")
assert credits_source_manifest["catalog_sha256"] == file_sha256(credits_path)
assert credits_source_manifest["cast_limit"] == 5
credits_payload = json.loads(credits_path.read_text(encoding="utf-8"))
credits = pd.DataFrame(credits_payload["movies"])
assert credits["movieId"].is_unique
assert set(credits["movieId"]) == set(movies["movieId"])
person_names = credits_payload["person_names"]
candidate_ids = np.sort(train_ratings["movieId"].unique())
coverage_rows = []
for scope, ids in [("full_catalog", movies["movieId"]), ("training_candidates", candidate_ids)]:
    subset = credits.set_index("movieId").loc[ids]
    for role in ["directors", "writers", "cast"]:
        present = subset[role].map(bool)
        coverage_rows.append({"scope": scope, "role": role, "movies": len(subset),
                              "with_credits": int(present.sum()), "coverage_pct": 100 * present.mean()})
coverage = pd.DataFrame(coverage_rows)
display(coverage.round(2))

Using verified subset: title.crew
Using verified subset: title.principals
Using verified subset: name.basics
Prepared 9,742 MovieLens credit records: C:\Users\clark\OneDrive\Documents\Projects\movie-recommendation-system\data\raw\imdb\movielens_credits.json


,scope,role,movies,with_credits,coverage_pct
0,full_catalog,directors,9742,9717,99.74
1,full_catalog,writers,9742,9560,98.13
2,full_catalog,cast,9742,9368,96.16
3,training_candidates,directors,6885,6872,99.81
4,training_candidates,writers,6885,6798,98.74
5,training_candidates,cast,6885,6693,97.21


## 2. Build fitting feedback and the shared candidate universe

Only positive **training** ratings form preference profiles. The seen mask includes low ratings too.
Popularity uses all training rating counts, then movie ID to break ties. No-credit profiles receive
the baseline hybrid in blends, or popularity in a content-only model; users are never dropped.

In [3]:
user_ids = np.sort(train_ratings["userId"].unique())
movie_ids = np.sort(train_ratings["movieId"].unique())
user_index = pd.Index(user_ids)
movie_index = pd.Index(movie_ids)
n_users, n_items = len(user_ids), len(movie_ids)

X = np.zeros((n_users, n_items), dtype=np.float64)
seen = np.zeros((n_users, n_items), dtype=bool)
train_u = user_index.get_indexer(train_ratings["userId"])
train_i = movie_index.get_indexer(train_ratings["movieId"])
seen[train_u, train_i] = True
positive_rows = train_ratings["rating"].ge(POSITIVE_THRESHOLD).to_numpy()
X[train_u[positive_rows], train_i[positive_rows]] = 1.0
liked_counts = X.sum(axis=1).astype(int)
popularity_counts = np.bincount(train_i, minlength=n_items)

validation_positives = validation_ratings.loc[
    validation_ratings["rating"].ge(POSITIVE_THRESHOLD)
]
positive_ids_by_user = {
    int(user_id): set(group["movieId"].astype(int))
    for user_id, group in validation_positives.groupby("userId", sort=True)
}
evaluation_user_ids = np.array(sorted(positive_ids_by_user), dtype=int)
if len(evaluation_user_ids) == 0:
    raise ValueError("No validation users have positive feedback.")
assert user_index.get_indexer(evaluation_user_ids).min() >= 0
assert X.sum() == positive_rows.sum()
assert seen.sum() == len(train_ratings)
assert popularity_counts.sum() == len(train_ratings)

# Keep chronological liked-item order to reproduce notebook 07's averaging.
liked_columns_by_user = [
    movie_index.get_indexer(
        train_ratings.loc[
            train_ratings["userId"].eq(user_id)
            & train_ratings["rating"].ge(POSITIVE_THRESHOLD), "movieId"
        ]
    )
    for user_id in user_ids
]
print("Binary feedback shape:", X.shape)
print("Training likes:", int(X.sum()))
print("Validation-positive users:", len(evaluation_user_ids))
print("Eligible users without training likes:",
      int((liked_counts[user_index.get_indexer(evaluation_user_ids)] == 0).sum()))

Binary feedback shape: (610, 6885)
Training likes: 29862
Validation-positive users: 594
Eligible users without training likes: 1


## 3. Construct separate feature groups

For each people-based group, an item has one feature per person ID. Retain people appearing in
at least **two training-catalog movies**, downweight ubiquitous people with inverse document frequency,
and L2-normalize each movie vector. This avoids features that cannot connect two candidate movies.
All vocabulary and frequency statistics are fitted on training candidates only.

Genre vectors are a control. Plot vectors reproduce notebook 04's log-TF TF-IDF settings.
Each score is the mean cosine similarity to the user's liked movies with usable features.
Director, writer, and cast scores are independently scaled before equal-weight combinations,
so five cast members do not automatically outweigh a director.

In [4]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer, normalize

def build_role_features(records, item_ids, role, min_movies=2):
    lookup = {int(row["movieId"]): row for row in records}
    labels = [sorted(set(lookup.get(int(item_id), {}).get(role, []))) for item_id in item_ids]
    encoder = MultiLabelBinarizer(sparse_output=True)
    matrix = encoder.fit_transform(labels).astype(np.float64).tocsr()
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    keep = counts >= min_movies
    matrix = matrix[:, keep]
    names = np.asarray(encoder.classes_)[keep]
    if matrix.shape[1]:
        idf = np.log((1 + len(item_ids)) / (1 + counts[keep])) + 1
        matrix = normalize(matrix.multiply(idf).tocsr(), norm="l2")
    return matrix.tocsr(), names

def score_features(feedback, features):
    feedback = csr_matrix(feedback)
    features = csr_matrix(features)
    covered = features.getnnz(axis=1) > 0
    usable_likes = np.asarray(feedback[:, covered].sum(axis=1)).ravel()
    scores = ((feedback @ features) @ features.T).toarray()
    np.divide(scores, usable_likes[:, None], out=scores, where=usable_likes[:, None] > 0)
    assert np.isfinite(scores).all()
    return scores, usable_likes

def scale_unseen(scores, seen_mask):
    scores = np.asarray(scores, dtype=float)
    if scores.shape != seen_mask.shape or not np.isfinite(scores).all() or (scores < 0).any():
        raise ValueError("Scores must be finite, nonnegative, and match the seen mask.")
    maximum = np.where(seen_mask, 0.0, scores).max(axis=1)
    result = np.zeros_like(scores)
    np.divide(scores, maximum[:, None], out=result, where=maximum[:, None] > 0)
    return result

def blend_with_fallback(base, content, weight, seen_mask):
    if not 0 <= weight <= 1:
        raise ValueError("Blend weight must lie in [0, 1].")
    base = scale_unseen(base, seen_mask)
    content = scale_unseen(content, seen_mask)
    result = (1 - weight) * base + weight * content
    missing = np.where(seen_mask, 0.0, content).max(axis=1) <= 0
    result[missing] = base[missing]
    return result

In [5]:
feature_matrices = {"cf": normalize(csr_matrix(X.T), norm="l2")}
feature_names = {}
for role in ["directors", "writers", "cast"]:
    feature_matrices[role], feature_names[role] = build_role_features(
        credits_payload["movies"], movie_ids, role
    )
candidate_movies = movies.set_index("movieId").loc[movie_ids]
genre_records = [{"movieId": int(movie_id),
                  "genres": [] if genres == "(no genres listed)" else genres.split("|")}
                 for movie_id, genres in candidate_movies["genres"].items()]
feature_matrices["genres"], feature_names["genres"] = build_role_features(
    genre_records, movie_ids, "genres", min_movies=1
)

plot_manifest_path = processed_dir / "plot_catalog_manifest.json"
if not plot_manifest_path.is_file():
    raise FileNotFoundError("Run notebook 04 first to prepare the plot catalog.")
plot_manifest = json.loads(plot_manifest_path.read_text(encoding="utf-8"))
assert plot_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
plot_path = processed_dir / "training_plot_catalog.csv"
assert file_sha256(plot_path) == plot_manifest["files"][plot_path.name]["sha256"]
plot_catalog = pd.read_csv(plot_path).set_index("movieId")
assert plot_catalog.index.is_unique and set(plot_catalog.index) == set(movie_ids)
plots = plot_catalog.loc[movie_ids, "plot"].fillna("")
plot_vectorizer = TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True, norm="l2")
feature_matrices["plots"] = plot_vectorizer.fit_transform(plots).tocsr()

component_scores, profile_counts = {}, {}
feature_rows = []
for role, matrix in feature_matrices.items():
    component_scores[role], profile_counts[role] = score_features(X, matrix)
    feature_rows.append({"group": role, "features": matrix.shape[1],
                         "candidate_movies_with_features": int((matrix.getnnz(axis=1) > 0).sum()),
                         "users_with_usable_likes": int((profile_counts[role] > 0).sum())})
feature_coverage = pd.DataFrame(feature_rows)
display(feature_coverage)

,group,features,candidate_movies_with_features,users_with_usable_likes
0,cf,610,4389,606
1,directors,1379,5475,605
2,writers,3060,5688,605
3,cast,4552,6375,605
4,genres,19,6880,606
5,plots,32042,5057,606


## 4. Compare controls, individual credit roles, and cumulative additions

First tune the CF + log-TF-IDF plot blend over the same four weights as notebook 07.
This is a freshly computed control, not a claim to reproduce notebook 07's eventual selected model.
Then add each credit group, directors + writers, and directors + writers + cast to that control.
Credit weights are fixed in advance at 0, 0.10, 0.25, 0.50, and 1.0; zero must reproduce the control.
Content-only people models and a CF + genres control help distinguish where any gain comes from.

The primary metric is mean per-user Recall@10. NDCG@10 measures ordering with binary relevance.
Both retain unavailable positives in their ideal denominator. Ties prefer the smaller added weight.
These comparisons reuse validation for tuning and selection, so improvements are exploratory.

In [6]:
def rank_unseen(scores, seen_row, counts, item_ids, k=10):
    scores = np.asarray(scores, dtype=float)
    item_ids = np.asarray(item_ids)
    if not (scores.shape == seen_row.shape == counts.shape == item_ids.shape):
        raise ValueError("Ranking arrays must have identical shapes.")
    if not np.isfinite(scores).all() or len(np.unique(item_ids)) != len(item_ids):
        raise ValueError("Ranking requires finite scores and unique item IDs.")
    candidates = np.flatnonzero(~seen_row)
    order = np.lexsort((item_ids[candidates], -counts[candidates], -scores[candidates]))
    return item_ids[candidates[order[:k]]]

def ranking_metrics(recommendations, positives, k=10):
    recommendations = list(recommendations)
    positives = set(positives)
    if not positives or len(recommendations) > k or len(set(recommendations)) != len(recommendations):
        raise ValueError("Require positives and up to k unique recommendations.")
    hits = np.array([item in positives for item in recommendations], dtype=float)
    discounts = 1 / np.log2(np.arange(2, len(recommendations) + 2))
    ideal = (1 / np.log2(np.arange(2, min(k, len(positives)) + 2))).sum()
    return int(hits.sum()), float(hits.sum() / len(positives)), float((hits * discounts).sum() / ideal)

def evaluate_scores(scores):
    rows = []
    for user_id in evaluation_user_ids:
        row = user_index.get_loc(user_id)
        recs = rank_unseen(scores[row], seen[row], popularity_counts, movie_ids, TOP_K)
        assert not set(recs) & set(movie_ids[seen[row]])
        positives = positive_ids_by_user[int(user_id)]
        hits, recall, ndcg = ranking_metrics(recs, positives, TOP_K)
        rows.append({"userId": int(user_id), "n_positives": len(positives),
                     "hits_at_10": hits, "recall_at_10": recall, "ndcg_at_10": ndcg})
    return pd.DataFrame(rows)

def mean_metrics(results):
    return {"users": len(results), "mean_recall_at_10": results["recall_at_10"].mean(),
            "mean_ndcg_at_10": results["ndcg_at_10"].mean()}

In [ ]:
BASE_WEIGHTS = [0.0, 0.10, 0.25, 0.50]
CREDIT_WEIGHTS = [0.0, 0.10, 0.25, 0.50, 1.0]
ROLE_SETS = {
    "directors": ("directors",),
    "writers": ("writers",),
    "cast": ("cast",),
    "directors_writers": ("directors", "writers"),
    "directors_writers_cast": ("directors", "writers", "cast"),
}
base_rows = []
for weight in BASE_WEIGHTS:
    scores = blend_with_fallback(component_scores["cf"], component_scores["plots"], weight, seen)
    base_rows.append({"plot_weight": weight, **mean_metrics(evaluate_scores(scores))})
base_grid = pd.DataFrame(base_rows).sort_values(
    ["mean_recall_at_10", "plot_weight"], ascending=[False, True]
)
chosen_plot_weight = float(base_grid.iloc[0]["plot_weight"])
base_scores = blend_with_fallback(component_scores["cf"], component_scores["plots"], chosen_plot_weight, seen)
base_results = evaluate_scores(base_scores)
cf_results = evaluate_scores(component_scores["cf"])

# Verify the existing popularity baseline if notebook 01's artifact is available.
popularity_results09 = evaluate_scores(np.zeros_like(base_scores))
saved_popularity = processed_dir / "popularity_validation_results.csv"
if saved_popularity.is_file():
    saved = pd.read_csv(saved_popularity)
    check_provenance(saved, "popularity")
    pd.testing.assert_series_equal(
        popularity_results09.set_index("userId")["recall_at_10"].sort_index(),
        saved.set_index("userId")["recall_at_10"].sort_index(), check_names=False, atol=1e-14, rtol=0,
    )
saved_cf = processed_dir / "item_based_validation_results.csv"
if saved_cf.is_file():
    saved = pd.read_csv(saved_cf)
    check_provenance(saved, "item-based CF")
    pd.testing.assert_series_equal(
        cf_results.set_index("userId")["recall_at_10"].sort_index(),
        saved.set_index("userId")["recall_at_10"].sort_index(), check_names=False, atol=1e-14, rtol=0,
    )

grid_rows, credit_signals = [], {}
best_results = {}
for label, roles in ROLE_SETS.items():
    signal = np.mean([scale_unseen(component_scores[role], seen) for role in roles], axis=0)
    credit_signals[label] = signal
    candidates = []
    for weight in CREDIT_WEIGHTS:
        results = evaluate_scores(blend_with_fallback(base_scores, signal, weight, seen))
        if weight == 0:
            pd.testing.assert_frame_equal(results, base_results)
        record = {"variant": label, "credit_weight": weight, **mean_metrics(results)}
        grid_rows.append(record)
        candidates.append((record["mean_recall_at_10"], -weight, results, weight))
    _, _, results, weight = max(candidates, key=lambda candidate: candidate[:2])
    best_results[label] = (weight, results)
credit_grid = pd.DataFrame(grid_rows)

summary_rows = [
    {"model": "Popularity", "credit_weight": 0.0, **mean_metrics(popularity_results09)},
    {"model": "Item-based CF", "credit_weight": 0.0, **mean_metrics(cf_results)},
    {"model": "CF + plots control", "credit_weight": 0.0, **mean_metrics(base_results)},
    {"model": "Genres only", "credit_weight": 0.0, **mean_metrics(evaluate_scores(component_scores["genres"]))},
]
for label in ["directors", "writers", "cast"]:
    summary_rows.append({"model": label + " only", "credit_weight": 1.0,
                         **mean_metrics(evaluate_scores(component_scores[label]))})
genre_rows = []
for weight in CREDIT_WEIGHTS:
    results = evaluate_scores(blend_with_fallback(component_scores["cf"], component_scores["genres"], weight, seen))
    genre_rows.append({"genre_weight": weight, **mean_metrics(results)})
genre_grid = pd.DataFrame(genre_rows).sort_values(["mean_recall_at_10", "genre_weight"], ascending=[False, True])
summary_rows.append({"model": "CF + genres (tuned)", "credit_weight": 0.0,
                     **genre_grid.iloc[0].to_dict()})
for label, (weight, results) in best_results.items():
    summary_rows.append({"model": "Control + " + label, "credit_weight": weight, **mean_metrics(results)})
summary = pd.DataFrame(summary_rows)
summary["recall_change_vs_control"] = summary["mean_recall_at_10"] - base_results["recall_at_10"].mean()
summary = summary.sort_values(["mean_recall_at_10", "model"], ascending=[False, True]).reset_index(drop=True)
print("Base plot-weight grid:")
display(base_grid.round(6))
print("Selected base plot weight:", chosen_plot_weight)
print("All metrics use the same", len(evaluation_user_ids), "validation users.")
display(summary.round(6))
display(credit_grid.round(6))

## 5. Inspect paired changes and readable recommendations

Compare each chosen credit blend with the exact same user's control result. Changes are paired
and descriptive; they are not an independent confirmation of a validation-selected winner.
Example users are chosen without looking at validation gains: use the lowest-ID eligible user
with a usable credit profile, then the next one. Shared credit names explain possible feature overlap,
not a causal explanation of the entire blended ranking.

In [ ]:
paired_rows = []
for label, (weight, results) in best_results.items():
    paired = results.merge(base_results, on="userId", suffixes=("_credits", "_control"), validate="one_to_one")
    assert paired["n_positives_credits"].equals(paired["n_positives_control"])
    delta = paired["recall_at_10_credits"] - paired["recall_at_10_control"]
    paired_rows.append({"variant": label, "credit_weight": weight,
                        "mean_recall_change": delta.mean(), "improved_users": int((delta > 0).sum()),
                        "worsened_users": int((delta < 0).sum()), "unchanged_users": int((delta == 0).sum())})
paired_summary = pd.DataFrame(paired_rows)
display(paired_summary.round(6))

winner_label = sorted(best_results, key=lambda label: (
    -best_results[label][1]["recall_at_10"].mean(), best_results[label][0], len(ROLE_SETS[label]), label
))[0]
winner_weight, winner_results = best_results[winner_label]
winner_scores = blend_with_fallback(base_scores, credit_signals[winner_label], winner_weight, seen)
credit_lookup = credits.set_index("movieId")
example_rows = []
eligible_rows = user_index.get_indexer(evaluation_user_ids)
has_credit_profile = sum(profile_counts[role] for role in ["directors", "writers", "cast"]) > 0
example_users = evaluation_user_ids[has_credit_profile[eligible_rows]][:2]
for user_id in example_users:
    row = user_index.get_loc(user_id)
    liked_ids = movie_ids[X[row] > 0]
    known_people = {role: {person for item in liked_ids for person in credit_lookup.loc[item, role]}
                    for role in ["directors", "writers", "cast"]}
    recs = rank_unseen(winner_scores[row], seen[row], popularity_counts, movie_ids, TOP_K)
    for rank, item in enumerate(recs, start=1):
        record = {"userId": int(user_id), "rank": rank, "movieId": int(item),
                  "title": candidate_movies.loc[item, "title"], "score": float(winner_scores[row, movie_index.get_loc(item)])}
        for role in known_people:
            shared = sorted(set(credit_lookup.loc[item, role]) & known_people[role])
            record["shared_" + role] = "; ".join(person_names.get(person, person) for person in shared)
        example_rows.append(record)
example_recommendations = pd.DataFrame(example_rows)
display(example_recommendations)
gain = winner_results["recall_at_10"].mean() - base_results["recall_at_10"].mean()
if winner_weight == 0 or gain <= 0:
    print("No tested credit blend improves validation Recall@10 over the control.")
else:
    print(f"Exploratory winner: {winner_label}, credit weight={winner_weight:.2f}, Recall@10 change={gain:+.6f}.")
print("Validation selection only; final test feedback has not been read.")

In [ ]:
credits_hash = file_sha256(credits_path)
def save_credit_results(frame, filename):
    saved = with_provenance(frame)
    saved["credits_catalog_sha256"] = credits_hash
    saved["plot_catalog_sha256"] = file_sha256(plot_path)
    saved.to_csv(output_dir / filename, index=False)

for frame, filename in [
    (coverage, "raw_credit_coverage.csv"), (feature_coverage, "feature_coverage.csv"),
    (base_grid, "base_plot_weight_grid.csv"), (genre_grid, "genre_weight_grid.csv"),
    (credit_grid, "credit_weight_grid.csv"), (summary, "validation_summary.csv"),
    (paired_summary, "paired_validation_changes.csv"),
    (example_recommendations, "example_recommendations.csv"),
    (base_results, "control_per_user.csv"), (cf_results, "cf_per_user.csv"),
]:
    save_credit_results(frame, filename)
for label, (weight, results) in best_results.items():
    save_credit_results(results.assign(credit_weight=weight), label + "_per_user.csv")
experiment_manifest = {
    "split_id": SPLIT_ID, "split_fingerprint": split_manifest["split_fingerprint"],
    "evaluation_split": "validation", "positive_threshold": POSITIVE_THRESHOLD, "k": TOP_K,
    "credits_catalog_sha256": credits_hash, "credits_sources": credits_source_manifest,
    "plot_catalog_sha256": file_sha256(plot_path),
    "minimum_candidate_movies_per_person": 2,
    "base_plot_weights": BASE_WEIGHTS, "credit_weights": CREDIT_WEIGHTS,
    "role_sets": ROLE_SETS, "chosen_plot_weight": chosen_plot_weight,
    "provisional_credit_variant": winner_label, "provisional_credit_weight": winner_weight,
    "selection_metric": "mean validation Recall@10; lower added weight breaks ties",
    "test_feedback_read": False,
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "scipy": scipy.__version__, "scikit_learn": sklearn.__version__},
    "files": {path.name: file_sha256(path) for path in sorted(output_dir.glob("*.csv"))},
}
(output_dir / "experiment_manifest.json").write_text(
    json.dumps(experiment_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("Saved validation experiment:", output_dir)

## Interpretation and next steps

- The summary reports whether credits improve retrieval over the CF + plots control, not whether
  a famous person makes a movie intrinsically better. Inspect both the mean and the paired changes.
- Metadata coverage and sparse histories can limit personalization. Missing credits remain zero
  features; they never shrink the candidate set or the validation-positive denominator.
- A current source snapshot, incomplete principal cast, related franchises, and many tuning
  comparisons can affect the apparent gain. Equal role weights and five cast entries are initial choices.
- Any winning blend is a **validation-selected hypothesis**. Do not reuse notebook 07's previously
  examined test results to tune this extension. Confirm with a new evaluation protocol or fresh data
  if that holdout has already influenced development. Keep this experiment separate from its frozen selection.